In [ ]:
import requests

headers = {
    'sec-ch-ua-platform': '"Windows"',
    'Referer': 'https://abe.web.geniussports.com/',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
}

response = requests.get(
    'https://hosted.dcd.shared.geniussports.com/embednf/ABE/es/competition/42034/schedule?&iurl=https%3A%2F%2Fabe.web.geniussports.com%2F%3Fp%3D9&_cc=1&_lc=1&_nv=1&_mf=1',
    headers=headers,
)

import pandas as pd
import json

if response.status_code == 200:
    print("¡Conexión exitosa!")

    try:
        data = response.json()

        print("Claves encontradas:", data.keys())

        print(json.dumps(data, indent=2)[:500])

    except:
        print("No es JSON, es HTML. Imprimiendo texto:")
        print(response.text[:500])
else:
    print(f"Error: {response.status_code}")

In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

html_content = data['html']
soup = BeautifulSoup(html_content, 'html.parser')

texto_completo = soup.get_text(separator="|", strip=True)
bloques_partidos = texto_completo.split("Fecha / Hora:")

print(f"{len(bloques_partidos) - 1} partidos encontrados.")

lista_final = []

prefijos_equipos = [
    "TEC MTY", "CEU ", "CETYS", "UANE", "UPAEP", "UDLAP", "ANAHUAC",
    "UP ", "UNIVERSIDAD", "AUTONOMA", "INTERAMERICANA", "UMAD", "UANL", "UACH"
] #Prefijos equipos par diferenciar de estadios

for bloque in bloques_partidos[1:]:
    partes = [p.strip() for p in bloque.split('|') if p.strip()]

    item = {"Fecha": "", "Hora": "", "Lugar": "", "Equipo_Local": None, "Puntos_Local": None, "Equipo_Visitante": None, "Puntos_Visitante": None}

    indices_a_ignorar = set()
    lugar_temporal = None # Guardamos el posible estadio aquí

    # Fecha y Lugar
    if len(partes) > 0:
        full_date = partes[0]
        indices_a_ignorar.add(0)
        if len(full_date) > 5 and ":" in full_date[-5:]:
            item["Fecha"] = full_date[:-5].strip(); item["Hora"] = full_date[-5:].strip()
        else: item["Fecha"] = full_date

    if "Pista:" in partes:
        idx = partes.index("Pista:")
        indices_a_ignorar.add(idx)

        if idx + 1 < len(partes):
            candidato_lugar = partes[idx+1]
            es_equipo_disfrazado = any(candidato_lugar.startswith(prefix) for prefix in prefijos_equipos)

            if not es_equipo_disfrazado:
                item["Lugar"] = candidato_lugar
                indices_a_ignorar.add(idx+1)
            else:
                lugar_temporal = candidato_lugar
                item["Lugar"] = "Sede por definir"

    # Equipos y Puntos
    basura = [
        "Final", "En directo", "Half time", "Prórroga", "OT", "Estadísticas completas",
        "-", ":", "vs", "VS", "Período", "Periodo", "Status", "Previa", "PREVIA",
        "Ver", "Reporte", "Link", "Q1", "Q2", "Q3", "Q4"
    ]

    equipos_crudos = []
    puntos = []

    for i, p in enumerate(partes):
        if i in indices_a_ignorar: continue
        es_fecha = ("2024" in p or "2025" in p or "2026" in p) and len(p) > 5
        es_boton = "Previa" in p or "Reporte" in p or "Link" in p

        if p not in basura and not es_fecha and not es_boton:
            if p.isdigit():
                puntos.append(p)
            elif len(p) >= 2 and p.lower() not in ['de', 'la', 'el', 'en']:
                equipos_crudos.append(p)

    # CORRECCIÓN DE CLONES
    equipos_limpios = []
    if equipos_crudos:
        equipos_limpios.append(equipos_crudos[0])
        for i in range(1, len(equipos_crudos)):
            if equipos_crudos[i] != equipos_crudos[i-1]:
                equipos_limpios.append(equipos_crudos[i])
            else:
                if lugar_temporal == equipos_crudos[i]:
                    item["Lugar"] = lugar_temporal

    if len(equipos_limpios) >= 1: item["Equipo_Local"] = equipos_limpios[0]
    if len(equipos_limpios) >= 2: item["Equipo_Visitante"] = equipos_limpios[1]

    if len(puntos) >= 1: item["Puntos_Local"] = puntos[0]
    if len(puntos) >= 2: item["Puntos_Visitante"] = puntos[1]

    lista_final.append(item)

# Crear DataFrame y Limpiar
df = pd.DataFrame(lista_final)
df_final = df.dropna(subset=['Equipo_Local'])
df_final = df_final.drop_duplicates(subset=['Fecha', 'Equipo_Local', 'Equipo_Visitante'])

# Auditoría
duplicados_mismo_equipo = df_final[df_final['Equipo_Local'] == df_final['Equipo_Visitante']]
print(f"Partidos donde Local == Visitante: {len(duplicados_mismo_equipo)} (Debería ser 0)")

vacios = df_final['Equipo_Visitante'].isna().sum()
print(f"Partidos sin rival: {vacios} (Debería ser 0)")

# Guardar
cols = ['Fecha', 'Hora', 'Lugar', 'Equipo_Local', 'Puntos_Local', 'Equipo_Visitante', 'Puntos_Visitante']
cols_existentes = [c for c in cols if c in df_final.columns]
df_final[cols_existentes].to_excel("Calendario_ABE_.xlsx", index=False)
print("'Calendario_ABE.xlsx' descargado")

In [ ]:
import pandas as pd
from datetime import datetime

try:
    df = pd.read_excel('Calendario_ABE.xlsx')
except:
    df = df_final.copy()

meses = {
    'ene': '01', 'feb': '02', 'mar': '03', 'abr': '04', 'may': '05', 'jun': '06',
    'jul': '07', 'ago': '08', 'sept': '09', 'sep': '09', 'oct': '10', 'nov': '11', 'dic': '12'
}

def parse_date(fecha_str):
    try:
        s = str(fecha_str).lower().replace('.', '').split()
        if len(s) >= 3:
            return pd.to_datetime(f"{s[2]}-{meses[s[1]]}-{s[0].zfill(2)}")
    except:
        return pd.NaT

df['Fecha_DT'] = df['Fecha'].apply(parse_date)

# FILTRO: Partidos hasta HOY
hoy = pd.Timestamp.now().normalize()
df_jugados = df[df['Fecha_DT'] <= hoy].copy()

df_jugados = df_jugados.sort_values(by='Fecha_DT', ascending=True).reset_index(drop=True)

df_jugados['Puntos_Local_Num'] = pd.to_numeric(df_jugados['Puntos_Local'], errors='coerce').fillna(0)
df_jugados['Puntos_Visitante_Num'] = pd.to_numeric(df_jugados['Puntos_Visitante'], errors='coerce').fillna(0)

print(f"DataFrame 'df_jugados' generado y ORDENADO por fecha antigua.")
print(f"Total de partidos jugados: {len(df_jugados)}")

# Verificación rápida de errores
errores = df_jugados[(df_jugados['Puntos_Local_Num'] == 0) | (df_jugados['Puntos_Visitante_Num'] == 0)]
print(f"Partidos pendientes de corrección: {len(errores)}")

# Imprimir las primeras filas para confirmar el orden
print("Primeros partidos (Los más antiguos):")
print(df_jugados[['Fecha', 'Equipo_Local', 'Puntos_Local', 'Puntos_Visitante', 'Equipo_Visitante']].head(103))

In [ ]:
errores = df_jugados[
    (df_jugados['Puntos_Local_Num'] == 0) |
    (df_jugados['Puntos_Visitante_Num'] == 0)
].copy()

errores = errores.sort_values(by='Fecha_DT', ascending=True)

print(f"Partidos por corregir encontrados: {len(errores)}")
print(errores[['Fecha', 'Equipo_Local', 'Puntos_Local', 'Puntos_Visitante', 'Equipo_Visitante']])

A continuación se corrigen los datos de los partidos de los cuales se pudo conseguir el resultados y se eliminan los que no.

In [ ]:
# Corrección directa por número de índice 68 (CETYS MEXICALI vs TEC MTY PUEBLA)
# Marcador: 61 (Local) - 74 (Visitante)
df_jugados.loc[68, ['Puntos_Local', 'Puntos_Visitante']] = [61, 74]

df_jugados.loc[68, ['Puntos_Local_Num', 'Puntos_Visitante_Num']] = [61, 74]

print("Partido Corregido")
print(df_jugados.loc[68, ['Fecha', 'Equipo_Local', 'Puntos_Local', 'Puntos_Visitante', 'Equipo_Visitante']])

In [ ]:
# Corrección directa por número de índice 167 (CETYS MEXICALI vs UP MEXICO)
# Marcador: 54 (Local) - 58 (Visitante)
df_jugados.loc[167, ['Puntos_Local', 'Puntos_Visitante']] = [54, 58]

df_jugados.loc[167, ['Puntos_Local_Num', 'Puntos_Visitante_Num']] = [54, 58]

print("Partido Corregido")
print(df_jugados.loc[167, ['Fecha', 'Equipo_Local', 'Puntos_Local', 'Puntos_Visitante', 'Equipo_Visitante']])

In [11]:
df_jugados = df_jugados.drop(83)
df_jugados = df_jugados.drop(246)
df_jugados = df_jugados.drop(288)

In [ ]:
columnas_finales = ['Equipo_Local', 'Puntos_Local', 'Puntos_Visitante', 'Equipo_Visitante']
df_final_jugados = df_jugados[columnas_finales].copy()

try:
    df_final_jugados['Puntos_Local'] = df_final_jugados['Puntos_Local'].astype(int)
    df_final_jugados['Puntos_Visitante'] = df_final_jugados['Puntos_Visitante'].astype(int)
except:
    print("Nota: No se pudieron convertir a enteros (quizás quedó algún texto), se mantienen como están.")

df_final_jugados = df_final_jugados.reset_index(drop=True)

print(f"DataFrame listo con {len(df_final_jugados)} partidos.")
print(df_final_jugados.head())

In [ ]:
df_final_jugados.to_csv("Resultados_ABE_Limpios.csv", index=False, encoding='utf-8-sig')

print("Archivo 'Resultados_ABE_Limpios.csv' guardado con éxito.")

In [ ]:
import pandas as pd
import numpy as np

equipos_unicos = sorted(pd.concat([df_final_jugados['Equipo_Local'], df_final_jugados['Equipo_Visitante']]).unique())

print(f"Equipos encontrados: {len(equipos_unicos)}")

matriz_ceros = np.zeros((len(df_final_jugados), len(equipos_unicos) * 2))

equipo_a_indice = {equipo: i for i, equipo in enumerate(equipos_unicos)}
n_equipos = len(equipos_unicos)

for idx, row in df_final_jugados.iterrows():
    nombre_local = row['Equipo_Local']
    puntos_local = row['Puntos_Local']

    nombre_visita = row['Equipo_Visitante']
    puntos_visita = row['Puntos_Visitante']

    col_idx_local = equipo_a_indice[nombre_local]
    matriz_ceros[idx, col_idx_local] = puntos_local

    col_idx_visita = equipo_a_indice[nombre_visita] + n_equipos
    matriz_ceros[idx, col_idx_visita] = puntos_visita

nombres_columnas = equipos_unicos + equipos_unicos
df_one_hot = pd.DataFrame(matriz_ceros, columns=nombres_columnas)

df_one_hot = df_one_hot.astype(int)

print("DataFrame generado con éxito.")
print(f"Dimensiones: {df_one_hot.shape} (Filas: Partidos, Cols: 2 veces cada equipo)")
print(df_one_hot.head())

df_one_hot.to_csv("Resultados_OneHot_38cols.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

equipos_unicos = sorted(pd.concat([df_final_jugados['Equipo_Local'], df_final_jugados['Equipo_Visitante']]).unique())

n_partidos = len(df_final_jugados)
n_equipos = len(equipos_unicos)
matriz = np.zeros((n_partidos, n_equipos * 2), dtype=int)

equipo_a_idx = {equipo: i for i, equipo in enumerate(equipos_unicos)}

for i, row in df_final_jugados.iterrows():
    col_local = equipo_a_idx[row['Equipo_Local']]
    matriz[i, col_local] = 1

    col_visita = equipo_a_idx[row['Equipo_Visitante']] + n_equipos
    matriz[i, col_visita] = 1

nombres_columnas = equipos_unicos + equipos_unicos
df_one_hot_binario = pd.DataFrame(matriz, columns=nombres_columnas)

print(f"Dimensiones: {df_one_hot.shape} (Filas: Partidos, Cols: 2 veces cada equipo)")
print(df_one_hot_binario.head())

df_one_hot_binario.to_csv("Partidos_OneHot_Binario.csv", index=False, encoding='utf-8-sig')

In [ ]:
df_one_hot_binario.to_csv("Partidos_OneHot_Binario.csv", index=False, encoding='utf-8-sig')

print("Archivo 'Partidos_OneHot_Binario.csv' generado exitosamente.")